# New DSAS workflow (v2 update)

This notebook builds an `NZCCDv2` **subset** from `NZCCDv1`, adds shoreline features for the AOIs picked up by the same new-data selection logic used in the new transect and uncertainty workflows, then runs DSAS-style calculations using the transects and `Total_UNCY` values produced by the two earlier notebooks.

Three things to know about the outputs:

- **They go in your own folder.** Set `RUN_OWNER` to your name and everything lands in `DataUpdatev2/<yourname>/`. Use the same value in all three notebooks. This is what stops two people who run the same AOI from overwriting each other.
- **Only the area you ran is kept.** NZCCDv1 rows outside the matched AOIs are dropped, so the file is your slice of the coast, not the whole country.
- **Filenames are tagged with your selection.** The tag comes from `search_mode`: `region`/`aoi` add the name, the `*_in_date_range` modes also add `since<YYYYMMDD>`.

Outputs (where `<tag>` is e.g. `Auckland_since20240718`):

- `DataUpdatev2/<yourname>/NZCCDv2_<tag>.shp` - shorelines for the AOIs in this run
- `DataUpdatev2/<yourname>/ratesv2_<tag>.shp` - transect-level DSAS rates + date/distance timeseries
- `DataUpdatev2/<yourname>/intersectsv2_<tag>.shp` - transect-shoreline intersection points + attributes
- `DataUpdatev2/<yourname>/new_dsas_exclusions_<tag>.csv` - shorelines left out of DSAS, and why

These per-area files are combined into the national dataset by `NZCCDv2_merge.ipynb`, which is run by the project maintainer.


In [5]:
%load_ext autotime
import warnings
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import statsmodels.api as sm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

SOURCE_DIR = Path("Data for testing")

# EDIT TARGET SHORELINES HERE
# See Part 6 of GETTING_STARTED.md; use the same RUN_OWNER and search criteria in all 3 notebooks.
RUN_OWNER = "catriona"
DATA_DIR = Path("DataUpdatev2") / RUN_OWNER
DATA_DIR.mkdir(parents=True, exist_ok=True)
V1_PATH = SOURCE_DIR / "NZCCDv1.shp"
TRANSECTS_PATH = DATA_DIR / "new_transects.shp"
UNCY_SUMMARY_PATH = DATA_DIR / "new_uncy_summary.csv"
# NZCCDv2 / rates / intersects filenames are built from the selection below, in the next cell

# The cutoff can be either a single date (e.g. "2024-07-18") or a date range as a
# two-item tuple/list (e.g. ("2024-07-18", "2024-08-18")).
cutoff_date = ("2024-07-18")
search_roots = [Path(r"Z:\MaxarImagery\HighFreq"), Path(r"Z:\Retrolens")]
search_mode = "date"  # 'date', 'aoi', 'aoi_in_date_range', 'region', or 'region_in_date_range'
target_aoi = "MedlandsBeach"
target_region = "Auckland"


def _norm(text):
    return ''.join(ch for ch in str(text).lower() if ch.isalnum())


def _coerce_cutoff_bounds(cutoff):
    if cutoff is None:
        raise ValueError("cutoff_date must not be None")

    if isinstance(cutoff, (tuple, list)):
        if len(cutoff) != 2:
            raise ValueError("cutoff_date range must be a two-item tuple/list")
        start = pd.Timestamp(cutoff[0]).normalize()
        end = pd.Timestamp(cutoff[1]).normalize()
        if start > end:
            start, end = end, start
        return start, end

    single = pd.Timestamp(cutoff).normalize()
    return single, None


def _matches_date(modified, cutoff_start, cutoff_end):
    if cutoff_end is None:
        return modified > cutoff_start
    return (modified >= cutoff_start) and (modified <= cutoff_end)


def _cutoff_label(cutoff_start, cutoff_end):
    if cutoff_end is None:
        return f"modified > {cutoff_start.date()}"
    return f"modified between {cutoff_start.date()} and {cutoff_end.date()}"


def _date_tag(cutoff):
    start, end = _coerce_cutoff_bounds(cutoff)
    if end is None:
        return f"since{start:%Y%m%d}"
    return f"since{start:%Y%m%d}to{end:%Y%m%d}"


def normalize_path(value):
    return str(value).replace('\\', '/').lower()


def pick_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def parse_date_from_stem(stem):
    text = str(stem)
    m = re.search(r"(\d{1,2}[A-Za-z]{3,4}\d{4})", text)
    if m:
        token = m.group(1).upper().replace("APRL", "APR").replace("SEPT", "SEP")
        for fmt in ("%d%b%Y", "%d%B%Y"):
            try:
                return pd.to_datetime(token, format=fmt)
            except Exception:
                pass
    m = re.search(r"(\d{4}-\d{2}-\d{2})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y-%m-%d")
        except Exception:
            pass
    m = re.search(r"(\d{8})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y%m%d")
        except Exception:
            pass
    return pd.NaT


def geom_hash(geom):
    if geom is None:
        return None
    try:
        return shapely.to_wkb(geom, hex=True)
    except Exception:
        return None

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 0 ns (started: 2026-08-11 15:36:16 +12:00)


In [6]:
# 1) Name the outputs after the selection, then load NZCCDv1 as the starting point
# Tagging filenames keeps two people working on different areas from writing to the same file.
def _slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "", str(text))

cutoff_start, cutoff_end = _coerce_cutoff_bounds(cutoff_date)
_date_tag = _date_tag(cutoff_date)
if search_mode == "date":
    OUTPUT_TAG = _date_tag
elif search_mode == "aoi":
    OUTPUT_TAG = _slug(target_aoi)
elif search_mode == "aoi_in_date_range":
    OUTPUT_TAG = f"{_slug(target_aoi)}_{_date_tag}"
elif search_mode == "region":
    OUTPUT_TAG = _slug(target_region)
elif search_mode == "region_in_date_range":
    OUTPUT_TAG = f"{_slug(target_region)}_{_date_tag}"
else:
    raise ValueError(f"Unknown search_mode: {search_mode}")

V2_PATH = DATA_DIR / f"NZCCDv2_{OUTPUT_TAG}.shp"
RATES_OUT = DATA_DIR / f"ratesv2_{OUTPUT_TAG}.shp"
POINTS_OUT = DATA_DIR / f"intersectsv2_{OUTPUT_TAG}.shp"
EXCLUSIONS_OUT = DATA_DIR / f"new_dsas_exclusions_{OUTPUT_TAG}.csv"

if not V1_PATH.exists():
    raise FileNotFoundError(f"Missing source dataset: {V1_PATH}")

v2 = gpd.read_file(V1_PATH)

print(f"Output tag: {OUTPUT_TAG}")
print(f"Will write: {V2_PATH.name} / {RATES_OUT.name} / {POINTS_OUT.name}")
print(f"Loaded NZCCDv1 rows: {len(v2):,}")
v2.head(2)


Output tag: since20240718
Will write: NZCCDv2_since20240718.shp / ratesv2_since20240718.shp / intersectsv2_since20240718.shp
Loaded NZCCDv1 rows: 19,666


,Region,Site,Digitiser,Scale,Notes,Source,CPS,Proxy,Photoscale,Georef_ER,Pixel_Er,Total_UNCY,USDate,SHLength,Date,ID,geometry
0,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,1.265403,2004-01-02,0.0,"LINESTRING Z (1728907.5 5916213.248 0, 1728868.341 5916181.498 0, 1728823.891 5916158.215 0, 1728765.154 5916132.815 0, 1728735.52 59161..."
1,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,0.307010,2004-01-02,1.0,"LINESTRING Z (1729067.838 5914779.733 0, 1729077.892 5914779.733 0, 1729089.534 5914776.028 0, 1729092.179 5914765.445 0, 1729082.654 59..."


time: 375 ms (started: 2026-08-11 15:36:16 +12:00)


In [7]:
# 2) Find new shoreline files using the same mode logic as new_transects/new_uncy
valid_modes = {'date', 'aoi', 'aoi_in_date_range', 'region', 'region_in_date_range'}
if search_mode not in valid_modes:
    raise ValueError(f"search_mode must be one of {sorted(valid_modes)}")
if search_mode in {'aoi', 'aoi_in_date_range'} and not str(target_aoi).strip():
    raise ValueError('target_aoi must be set when using AOI-based modes')
if search_mode in {'region', 'region_in_date_range'} and not str(target_region).strip():
    raise ValueError('target_region must be set when using region-based modes')

cutoff_start, cutoff_end = _coerce_cutoff_bounds(cutoff_date)
target_aoi_norm = _norm(target_aoi)
target_region_norm = _norm(target_region)
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        if len(shp.parts) < 5 or shp.parts[-2].lower() != 'shorelines':
            continue

        region = shp.parts[-4]
        aoi = shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]

        matches_aoi = target_aoi_norm in {_norm(aoi), _norm(stem_aoi)}
        matches_region = target_region_norm == _norm(region)
        matches_date = _matches_date(modified, cutoff_start, cutoff_end)

        include = False
        if search_mode == 'date':
            include = matches_date
        elif search_mode == 'aoi':
            include = matches_aoi
        elif search_mode == 'aoi_in_date_range':
            include = matches_aoi and matches_date
        elif search_mode == 'region':
            include = matches_region
        elif search_mode == 'region_in_date_range':
            include = matches_region and matches_date

        if include:
            records.append({
                'region': region,
                'aoi': aoi,
                'shoreline_path': str(shp),
                'modified': modified,
            })

new_shorelines = pd.DataFrame(records)
if new_shorelines.empty:
    raise ValueError('No shoreline files matched the selected search criteria')

new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)
target_aois = new_shorelines[['region', 'aoi']].drop_duplicates().reset_index(drop=True)

print(f"Matched new shoreline files: {len(new_shorelines)}")
print(f"Target AOIs for trimming NZCCDv1: {len(target_aois)}")
new_shorelines.head(20)

Matched new shoreline files: 181
Target AOIs for trimming NZCCDv1: 58


,region,aoi,shoreline_path,modified
0,Auckland,Hobsonville,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,2026-08-05 22:49:58.488281488
1,Auckland,KarekareBethells,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JUL2024.shp,2026-07-30 05:13:29.477445602
2,Auckland,KarekareBethells,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_19MAR2023.shp,2026-07-30 05:13:29.679181576
3,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_09MAR2011.shp,2025-06-16 01:42:47.641532183
4,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_07FEB1982.shp,2025-10-23 23:45:07.069980383
5,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_16DEC1976.shp,2025-10-23 23:45:07.408631086
6,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_02MAY1996.shp,2025-10-23 23:45:07.690840006
7,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_14OCT1966.shp,2025-10-23 23:45:07.957997799
8,Auckland,ManukapuaIsland,Z:\MaxarImagery\HighFreq\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_20JUL2019.shp,2026-08-11 03:23:19.499143600
9,Auckland,ManukapuaIsland,Z:\MaxarImagery\HighFreq\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_08AUG2022.shp,2026-08-11 03:23:19.871378183


time: 4min 42s (started: 2026-08-11 15:36:16 +12:00)


In [8]:
# 2b) Trim NZCCDv1 down to the areas this run actually covers
# Rows outside the matched AOIs are dropped, so the NZCCDv2 written below holds only
# the coastline this run is responsible for and can be merged with other people's runs later.
region_col = pick_col(v2, ["Region"])
location_col = pick_col(v2, ["Location", "Site"])
if region_col is None or location_col is None:
    raise ValueError("NZCCDv1 has no Region/Location(Site) columns, so it cannot be trimmed to the target AOIs")

target_pairs = {(_norm(r.region), _norm(r.aoi)) for r in target_aois.itertuples(index=False)}
v2_pairs = list(zip(v2[region_col].map(_norm), v2[location_col].map(_norm)))
keep_mask = pd.Series([pair in target_pairs for pair in v2_pairs], index=v2.index)

kept_pairs = {pair for pair, keep in zip(v2_pairs, keep_mask) if keep}
missing_pairs = sorted(target_pairs - kept_pairs)

dropped = int((~keep_mask).sum())
v2 = v2[keep_mask].reset_index(drop=True)

# NZCCDv1 calls the AOI column "Site"; the merge/dedupe steps below expect "Location"
if "Location" not in v2.columns:
    v2["Location"] = v2[location_col]

print(f"Kept {len(v2):,} NZCCDv1 rows across {len(kept_pairs)} AOI(s); dropped {dropped:,} rows outside this run")
if missing_pairs:
    print(f"No NZCCDv1 history for {len(missing_pairs)} target AOI(s) - those shorelines come from the drive only:")
    for region, aoi in missing_pairs:
        print(f"  - {region} / {aoi}")
if v2.empty:
    print("WARNING: no NZCCDv1 rows matched. Check that Region/Site spelling in NZCCDv1 matches the drive folder names.")


Kept 3,313 NZCCDv1 rows across 38 AOI(s); dropped 16,353 rows outside this run
No NZCCDv1 history for 20 target AOI(s) - those shorelines come from the drive only:
  - auckland / hobsonville
  - auckland / manukapuaisland
  - auckland / ngataringa
  - auckland / omokoitibay
  - auckland / orongopoint
  - auckland / pollenisland
  - auckland / shellybeach
  - auckland / shoalbay
  - auckland / soldiersbay
  - auckland / teatatu
  - auckland / tehakonoclarksbay
  - canterbury / kaikoura
  - canterbury / timaru
  - manawatuwhanganui / whanganuisouth
  - northland / frenchmansbay
  - northland / pouto
  - northland / tinopai
  - wellington / paekakariki
  - wellington / paraparaumu
  - wellington / tehorobeach
time: 125 ms (started: 2026-08-11 15:40:59 +12:00)


In [9]:
# 3) Build full AOI shoreline set (all dates) and append missing rows into NZCCDv2
if not UNCY_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing uncertainty summary: {UNCY_SUMMARY_PATH}")

uncy_summary = pd.read_csv(UNCY_SUMMARY_PATH)
required_uncy_cols = {'path', 'total_uncy_mean'}
missing_uncy_cols = required_uncy_cols - set(uncy_summary.columns)
if missing_uncy_cols:
    raise ValueError(f"{UNCY_SUMMARY_PATH} is missing columns: {sorted(missing_uncy_cols)}")

uncy_summary = uncy_summary.copy()
uncy_summary['path_norm'] = uncy_summary['path'].map(normalize_path)
uncy_summary['total_uncy_mean'] = pd.to_numeric(uncy_summary['total_uncy_mean'], errors='coerce')
uncy_map = (
    uncy_summary.dropna(subset=['path_norm', 'total_uncy_mean'])
    .drop_duplicates('path_norm', keep='last')
    .set_index('path_norm')['total_uncy_mean']
    .to_dict()
)

new_file_paths_norm = set(new_shorelines['shoreline_path'].map(normalize_path).tolist())
all_aoi_rows = []
exclusion_rows = []

for row in target_aois.itertuples(index=False):
    region = row.region
    aoi = row.aoi

    shoreline_files = []
    for root in search_roots:
        shoreline_dir = root / region / aoi / 'Shorelines'
        if shoreline_dir.exists():
            shoreline_files.extend(sorted(shoreline_dir.glob('*.shp')))

    for shp in shoreline_files:
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        g = gpd.read_file(shp)
        if g.empty:
            continue

        if g.crs is not None and v2.crs is not None and str(g.crs) != str(v2.crs):
            g = g.to_crs(v2.crs)

        date_col = pick_col(g, ['Date'])
        if date_col is None:
            g['Date'] = parse_date_from_stem(shp.stem)
        else:
            g['Date'] = pd.to_datetime(g[date_col], errors='coerce')

        shp_norm = normalize_path(shp)
        g['Total_UNCY'] = uncy_map.get(shp_norm, np.nan)

        g['Region'] = region
        g['Location'] = aoi
        g['SourceFile'] = str(shp)
        g['GeomHash'] = g.geometry.apply(geom_hash)

        uncy_series = pd.to_numeric(g['Total_UNCY'], errors='coerce')
        eligible_mask = g['Date'].notna() & uncy_series.notna() & (uncy_series > 0)
        excluded_rows = int((~eligible_mask).sum())
        if excluded_rows > 0:
            reasons = []
            missing_date_rows = int(g['Date'].isna().sum())
            missing_uncy_rows = int(uncy_series.isna().sum())
            non_positive_uncy_rows = int((uncy_series.notna() & (uncy_series <= 0)).sum())
            if missing_uncy_rows > 0:
                reasons.append('missing Total_UNCY')
            if non_positive_uncy_rows > 0:
                reasons.append('non-positive Total_UNCY')
            if missing_date_rows > 0:
                reasons.append('missing Date')
            exclusion_rows.append({
                'filename': str(shp),
                'region': region,
                'aoi': aoi,
                'is_new_shoreline': shp_norm in new_file_paths_norm,
                'total_rows': int(len(g)),
                'excluded_rows': excluded_rows,
                'included_rows': int(eligible_mask.sum()),
                'reason': '; '.join(reasons),
            })

        keep_cols = [c for c in v2.columns if c in g.columns]
        if 'Date' not in keep_cols:
            keep_cols.append('Date')
        if 'Total_UNCY' not in keep_cols:
            keep_cols.append('Total_UNCY')
        if 'Region' not in keep_cols:
            keep_cols.append('Region')
        if 'Location' not in keep_cols:
            keep_cols.append('Location')
        keep_cols.extend([c for c in ['SourceFile', 'GeomHash', 'geometry'] if c not in keep_cols and c in g.columns])

        all_aoi_rows.append(g[keep_cols].copy())

if len(all_aoi_rows) == 0:
    raise ValueError('No shoreline features found for target AOIs.')

incoming = gpd.GeoDataFrame(pd.concat(all_aoi_rows, ignore_index=True), crs=v2.crs)
if 'GeomHash' not in v2.columns:
    v2['GeomHash'] = v2.geometry.apply(geom_hash)
if 'Region' not in v2.columns:
    v2['Region'] = pd.NA
if 'Location' not in v2.columns:
    v2['Location'] = pd.NA
if 'Date' in v2.columns:
    v2['Date'] = pd.to_datetime(v2['Date'], errors='coerce')
else:
    v2['Date'] = pd.NaT
if 'Total_UNCY' in v2.columns:
    v2['Total_UNCY'] = pd.to_numeric(v2['Total_UNCY'], errors='coerce')
else:
    v2['Total_UNCY'] = np.nan

for col in incoming.columns:
    if col not in v2.columns:
        v2[col] = pd.NA
for col in v2.columns:
    if col not in incoming.columns:
        incoming[col] = pd.NA
incoming = incoming[v2.columns]

v2_key = pd.DataFrame({
    'Region': v2['Region'].astype(str).map(_norm),
    'Location': v2['Location'].astype(str).map(_norm),
    'Date': pd.to_datetime(v2['Date'], errors='coerce').astype(str),
    'GeomHash': v2['GeomHash'].astype(str),
})
incoming_key = pd.DataFrame({
    'Region': incoming['Region'].astype(str).map(_norm),
    'Location': incoming['Location'].astype(str).map(_norm),
    'Date': pd.to_datetime(incoming['Date'], errors='coerce').astype(str),
    'GeomHash': incoming['GeomHash'].astype(str),
})

existing = set(map(tuple, v2_key[['Region', 'Location', 'Date', 'GeomHash']].itertuples(index=False, name=None)))
is_new = incoming_key[['Region', 'Location', 'Date', 'GeomHash']].apply(tuple, axis=1).map(lambda t: t not in existing)
to_add = incoming[is_new].copy()

v2_updated = gpd.GeoDataFrame(pd.concat([v2, to_add], ignore_index=True), crs=v2.crs)
v2_updated.to_file(V2_PATH)

if len(exclusion_rows) == 0:
    dsas_exclusions = pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason'])
else:
    dsas_exclusions = pd.DataFrame(exclusion_rows).drop_duplicates().reset_index(drop=True)

print(f"Loaded uncertainty values for {len(uncy_map):,} shoreline files from {UNCY_SUMMARY_PATH}")
print(f"Added {len(to_add):,} shoreline rows to NZCCDv2")
print(f"NZCCDv2 total rows: {len(v2_updated):,}")
print(f"Shoreline files with pre-DSAS exclusions: {dsas_exclusions['filename'].nunique() if len(dsas_exclusions) > 0 else 0}")
display(dsas_exclusions.head(20))
v2_updated[['Region', 'Location', 'Date', 'Total_UNCY']].tail(10)

Loaded uncertainty values for 78 shoreline files from DataUpdatev2\catriona\new_uncy_summary.csv
Added 4,901 shoreline rows to NZCCDv2
NZCCDv2 total rows: 8,214
Shoreline files with pre-DSAS exclusions: 591


c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value '010200008031000000C0E0F97F8B613A4182C5DF4F8D915641000000000000000050366C5764613A4128BBDE5F859156410000000000000000942F33E437613A41B6D5BB8D7F9156410000000000000000E0AF5E27FD603A410B67213479915641000000000000000068AB3885DF603A41B62F540776915641000000000000000098C9DD2FB5603A41835564046F9156410000000000000000083665B8AC603A411E47EC846A915641000000000000000028B7EE20B4603A4116F2FC79679156410000000000000000F400AB5CB8603A41F0EEB7C9619156410000000000000000D8CBF000C4603A410DF7A5E85A9156410000000000000000803A8B5ACA603A4144ADE9AC569156410000000000000000D8F8BC3DBF603A41F6F7D7C751915641000000000000000060B901F2C2603A4149893D6E4B9156410000000000000000D8CBF000C4603A412D08B405449156410000000000000000D8CBF000C4603A41FFAF7F4A3B9156410000000000000000F07946A6C6603A41F105A134359156410000000

,filename,region,aoi,is_new_shoreline,total_rows,excluded_rows,included_rows,reason
0,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,Auckland,Hobsonville,True,78,78,0,missing Total_UNCY
1,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_01MAR2015.shp,Auckland,KarekareBethells,False,15,15,0,missing Total_UNCY
2,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_03JAN2011.shp,Auckland,KarekareBethells,False,8,8,0,missing Total_UNCY
3,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JAN2017.shp,Auckland,KarekareBethells,False,19,19,0,missing Total_UNCY
4,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JAN2022.shp,Auckland,KarekareBethells,False,19,19,0,missing Total_UNCY
5,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JUL2024.shp,Auckland,KarekareBethells,True,6,6,0,missing Total_UNCY
6,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_05APR2016.shp,Auckland,KarekareBethells,False,6,6,0,missing Total_UNCY
7,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_07APRIL2010.shp,Auckland,KarekareBethells,False,9,9,0,missing Total_UNCY
8,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_19MAR2023.shp,Auckland,KarekareBethells,True,6,6,0,missing Total_UNCY
9,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_20SEP2008.shp,Auckland,KarekareBethells,False,9,9,0,missing Total_UNCY


,Region,Location,Date,Total_UNCY
8204,WestCoast,GreymouthSouth,2023-11-17,NaN
8205,WestCoast,GreymouthSouth,2023-11-17,NaN
8206,WestCoast,GreymouthSouth,2023-11-17,NaN
8207,WestCoast,GreymouthSouth,2023-11-17,NaN
8208,WestCoast,GreymouthSouth,2023-11-17,NaN
8209,WestCoast,GreymouthSouth,1988-02-18,NaN
8210,WestCoast,GreymouthSouth,1988-02-18,NaN
8211,WestCoast,GreymouthSouth,1988-02-18,NaN
8212,WestCoast,GreymouthSouth,1945-05-21,NaN
8213,WestCoast,GreymouthSouth,1945-05-21,NaN


time: 1min 26s (started: 2026-08-11 15:40:59 +12:00)


In [10]:
# 4) DSAS calculations using new transects and updated NZCCDv2 shorelines
if not TRANSECTS_PATH.exists():
    raise FileNotFoundError(f"Missing transects: {TRANSECTS_PATH}")

transects = gpd.read_file(TRANSECTS_PATH)
uid_col = pick_col(transects, ['Unique_ID', 'UniqueID'])
if uid_col is None:
    raise ValueError('Transects must include Unique_ID or UniqueID')
transects = transects.rename(columns={uid_col: 'Unique_ID'}).set_index('Unique_ID')
if transects.crs is None:
    transects = transects.set_crs(2193, allow_override=True)
else:
    transects = transects.to_crs(2193)

# Ensure one transect geometry per Unique_ID.
if transects.index.duplicated().any():
    dup_count = int(transects.index.duplicated().sum())
    print(f"Dropping {dup_count:,} duplicate transect rows by Unique_ID (keeping first geometry)")
    transects = transects[~transects.index.duplicated(keep='first')].copy()

shore = gpd.read_file(V2_PATH)
if shore.crs is None:
    shore = shore.set_crs(2193, allow_override=True)
else:
    shore = shore.to_crs(2193)

region_col = pick_col(shore, ['Region', 'region'])
aoi_col = pick_col(shore, ['Location', 'AOI', 'aoi', 'location'])
date_col = pick_col(shore, ['Date', 'date'])
uncy_col = pick_col(shore, ['Total_UNCY', 'total_uncy', 'new_Total_UNCY'])
sourcefile_col = pick_col(shore, ['SourceFile', 'sourcefile'])

if region_col is None or aoi_col is None or date_col is None:
    raise ValueError('NZCCDv2 must include Region/Location(Date) columns for DSAS')

shore = shore.rename(columns={region_col: 'Region', aoi_col: 'AOI', date_col: 'Date'})
shore = shore.loc[:, ~shore.columns.duplicated()].copy()
shore = shore.set_geometry('geometry')
shore['Date'] = pd.to_datetime(shore['Date'], errors='coerce')
if uncy_col is None:
    shore['Total_UNCY'] = np.nan
else:
    shore['Total_UNCY'] = pd.to_numeric(shore[uncy_col], errors='coerce')
if sourcefile_col is None:
    shore['SourceFile'] = pd.NA
else:
    shore['SourceFile'] = shore[sourcefile_col].astype(str)

target_key = set(target_aois.apply(lambda r: (_norm(r.region), _norm(r.aoi)), axis=1).tolist())
shore = shore[shore.apply(lambda r: (_norm(r['Region']), _norm(r['AOI'])) in target_key, axis=1)].copy()
shore = shore[shore.geometry.notna()].copy()
shore['dsas_eligible'] = shore['Date'].notna() & shore['Total_UNCY'].notna() & (shore['Total_UNCY'] > 0)

def to_point_or_empty(geom, transect_origin):
    if geom is None or geom.is_empty:
        return shapely.Point()
    gt = geom.geom_type
    if gt == 'Point':
        return geom
    if gt == 'MultiPoint':
        pts = list(geom.geoms)
        if len(pts) == 0:
            return shapely.Point()
        pts = sorted(pts, key=lambda p: p.distance(transect_origin))
        return pts[0]
    try:
        p = shapely.get_point(geom, 0)
        return p if p is not None else shapely.Point()
    except Exception:
        return shapely.Point()

def intersect_or_empty(geom, line):
    if geom is None:
        return shapely.GeometryCollection()
    try:
        return geom.intersection(line)
    except Exception:
        return shapely.GeometryCollection()

def process_transect(unique_id):
    transect = transects.geometry.loc[unique_id]
    tran_origin = shapely.get_point(transect, -1)

    local = shore.copy()
    intersections = [intersect_or_empty(geom, transect) for geom in local.geometry.values]
    local['intersect_raw'] = intersections
    local['intersect_point'] = [to_point_or_empty(g, tran_origin) for g in intersections]

    # Promote intersection points to a GeoSeries so geometric vector ops are available.
    point_gs = gpd.GeoSeries(local['intersect_point'], index=local.index, crs=shore.crs)
    local = local[~point_gs.is_empty].copy()
    point_gs = point_gs.loc[local.index]
    local = local[local['dsas_eligible']].sort_values('Date')
    point_gs = point_gs.loc[local.index]

    if len(local) < 3:
        return None, None

    local['YearsSinceBase'] = (local['Date'] - local['Date'].min()).dt.days / 365.25
    local['Distance'] = point_gs.distance(tran_origin)

    lr = sm.OLS(local['Distance'], sm.add_constant(local['YearsSinceBase'])).fit()
    lr_low, lr_high = lr.conf_int(alpha=0.1).loc['YearsSinceBase']
    lci = (lr_high - lr_low) / 2

    wlr = sm.WLS(local['Distance'], sm.add_constant(local['YearsSinceBase']), weights=1 / (local['Total_UNCY'] ** 2)).fit()
    wlr_low, wlr_high = wlr.conf_int(alpha=0.1).loc['YearsSinceBase']
    wci = (wlr_high - wlr_low) / 2

    duration = (local['Date'].max() - local['Date'].min()).days / 365.25
    if duration <= 0:
        return None, None

    nsm = -(local['Distance'].iloc[0] - local['Distance'].iloc[-1])
    sce = point_gs.apply(lambda p: point_gs.distance(p).max()).max()

    rate_row = {
        'UniqueID': unique_id,
        'Region': local['Region'].astype(str).value_counts().idxmax(),
        'AOI': local['AOI'].astype(str).value_counts().idxmax(),
        'Start_date': str(local['Date'].min().date()),
        'End_date': str(local['Date'].max().date()),
        'Duration': round(duration),
        'ShrCount': len(local),
        'NSM': round(nsm, 2),
        'SCE': round(sce, 2),
        'EPR': round(nsm / duration, 2),
        'EPRunc': round(np.sqrt(local['Total_UNCY'].iloc[0] ** 2 + local['Total_UNCY'].iloc[-1] ** 2) / duration, 2),
        'LRR': round(lr.params['YearsSinceBase'], 2),
        'LRI': round(lr.params['const'], 2),
        'LCI': round(lci, 2),
        'LSE': round(np.sqrt(lr.mse_resid), 2),
        'LR2': round(lr.rsquared, 2),
        'WLR': round(wlr.params['YearsSinceBase'], 2),
        'WLI': round(wlr.params['const'], 2),
        'WCI': round(wci, 2),
        'WSE': round(np.sqrt(wlr.mse_resid), 2),
        'WR2': round(wlr.rsquared, 2),
        'Dates': local['Date'].dt.strftime('%Y-%m-%d').tolist(),
        'Distances': local['Distance'].round(2).tolist(),
        'geometry': transect,
    }

    point_rows = local[['Region', 'AOI', 'Date', 'Distance', 'Total_UNCY', 'SourceFile', 'intersect_point']].copy()
    point_rows['Unique_ID'] = unique_id
    point_rows['YearsSinceBase'] = local['YearsSinceBase']
    point_rows['NSM'] = round(nsm, 2)
    point_rows['EPR'] = round(nsm / duration, 2)
    point_rows['LRR'] = round(lr.params['YearsSinceBase'], 2)
    point_rows['WLR'] = round(wlr.params['YearsSinceBase'], 2)
    point_rows = point_rows.rename(columns={'intersect_point': 'geometry'})

    return rate_row, point_rows

rate_rows = []
point_frames = []
for uid in tqdm(transects.index.tolist()):
    rate_row, point_rows = process_transect(uid)
    if rate_row is None:
        continue
    rate_rows.append(rate_row)
    point_frames.append(point_rows)

if len(rate_rows) == 0:
    raise ValueError('No transects produced DSAS statistics (need at least 3 eligible shoreline intersections per transect).')

rates = gpd.GeoDataFrame(rate_rows, crs=transects.crs)
points = gpd.GeoDataFrame(pd.concat(point_frames, ignore_index=True), crs=transects.crs)
points['Date'] = pd.to_datetime(points['Date'], errors='coerce')

# Add post-DSAS exclusions for eligible shoreline files that never made it into DSAS outputs.
if 'dsas_exclusions' not in globals():
    dsas_exclusions = pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason'])

eligible_files = set(
    shore.loc[shore['dsas_eligible'] & shore['SourceFile'].notna() & (shore['SourceFile'].astype(str) != 'nan'), 'SourceFile']
    .astype(str)
    .unique()
    .tolist()
)
used_files = set(points['SourceFile'].dropna().astype(str).unique().tolist())
not_used_files = sorted(eligible_files - used_files)

if len(not_used_files) > 0:
    extra_exclusions = pd.DataFrame({
        'filename': not_used_files,
        'region': [pd.NA] * len(not_used_files),
        'aoi': [pd.NA] * len(not_used_files),
        'is_new_shoreline': [pd.NA] * len(not_used_files),
        'total_rows': [pd.NA] * len(not_used_files),
        'excluded_rows': [pd.NA] * len(not_used_files),
        'included_rows': [pd.NA] * len(not_used_files),
        'reason': ['eligible shoreline not used in final DSAS results (no successful transect intersections)'] * len(not_used_files),
    })
    dsas_exclusions = pd.concat([dsas_exclusions, extra_exclusions], ignore_index=True).drop_duplicates()

display(rates.head(10))
display(points.head(10))
print(f"Rates rows: {len(rates):,}")
print(f"Point rows: {len(points):,}")
print(f"DSAS exclusions rows: {len(dsas_exclusions):,}")

Dropping 4,190 duplicate transect rows by Unique_ID (keeping first geometry)


  0%|          | 0/36484 [00:00<?, ?it/s]

,UniqueID,Region,AOI,Start_date,End_date,Duration,ShrCount,NSM,SCE,EPR,EPRunc,LRR,LRI,LCI,LSE,LR2,WLR,WLI,WCI,WSE,WR2,Dates,Distances,geometry
0,100632729406,Auckland,KarekareBethells,1960-08-19,2021-03-21,61,9,-20.13,20.30,-0.33,0.16,-0.30,252.33,0.16,4.74,0.63,-0.30,252.12,0.16,1.25,0.64,"[1960-08-19, 1988-03-20, 2004-01-02, 2011-01-03, 2011-09-23, 2015-03-01, 2016-04-05, 2017-01-04, 2021-03-21]","[251.7, 240.07, 248.27, 237.16, 241.81, 237.36, 231.39, 232.09, 231.57]","LINESTRING (1730695.783 5908585.729, 1730438.347 5908891.877)"
1,100632730071,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-38.94,40.37,-0.48,0.06,-0.46,268.47,0.12,5.98,0.76,-0.51,271.54,0.16,2.20,0.67,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[263.07, 257.49, 253.13, 248.48, 241.5, 243.91, 245.69, 241.82, 238.91, 242.34, 242.5, 226.14, 230.03, 228.46, 228.12, 222.7, 224.14]","LINESTRING (1730703.437 5908592.165, 1730446.001 5908898.313)"
2,100632730919,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-44.29,46.03,-0.54,0.06,-0.51,272.42,0.15,7.79,0.70,-0.56,275.17,0.20,2.66,0.63,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[266.31, 256.52, 255.53, 256.89, 243.37, 243.02, 249.08, 242.89, 242.51, 241.74, 245.16, 224.57, 228.79, 226.0, 228.51, 220.28, 222.03]","LINESTRING (1730711.091 5908598.601, 1730453.654 5908904.749)"
3,100632732206,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-48.51,49.72,-0.59,0.06,-0.56,277.91,0.17,9.01,0.68,-0.62,280.74,0.21,2.91,0.63,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[270.74, 260.06, 257.27, 265.29, 245.27, 244.12, 250.55, 245.62, 246.23, 239.86, 248.86, 223.23, 230.45, 226.32, 227.97, 221.02, 222.24]","LINESTRING (1730718.721 5908605.018, 1730461.331 5908911.204)"
4,100632733836,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-47.97,50.60,-0.59,0.06,-0.57,280.58,0.18,9.28,0.67,-0.64,284.49,0.24,3.21,0.60,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[272.79, 264.68, 259.0, 264.4, 247.81, 246.48, 252.6, 254.14, 249.37, 245.93, 247.62, 224.84, 233.12, 227.1, 228.9, 222.19, 224.83]","LINESTRING (1730726.375 5908611.454, 1730468.985 5908917.64)"
5,100632735480,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-43.65,45.71,-0.53,0.06,-0.54,278.54,0.15,7.93,0.72,-0.59,280.45,0.19,2.53,0.67,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[269.23, 264.29, 262.17, 262.47, 250.36, 246.83, 252.81, 245.57, 243.99, 240.49, 247.16, 226.44, 235.1, 228.22, 232.4, 223.53, 225.58]","LINESTRING (1730734.051 5908617.909, 1730476.615 5908924.057)"
6,100632736992,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-39.89,39.89,-0.49,0.06,-0.43,271.80,0.17,8.62,0.57,-0.49,275.02,0.22,3.03,0.50,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[264.05, 256.86, 260.08, 260.54, 251.37, 247.74, 251.84, 252.64, 245.68, 246.1, 250.82, 231.75, 236.6, 231.56, 231.32, 224.85, 224.16]","LINESTRING (1730741.705 5908624.345, 1730484.269 5908930.493)"
7,100632738500,Auckland,KarekareBethells,1940-04-14,2022-01-04,82,17,-30.31,32.00,-0.37,0.06,-0.33,262.08,0.18,9.36,0.40,-0.39,264.83,0.23,3.09,0.38,"[1940-04-14, 1960-08-19, 1980-10-24, 1988-03-20, 2004-01-02, 2006-08-23, 2008-09-20, 2010-04-07, 2011-01-03, 2011-09-23, 2013-03-25, 201...","[253.65, 248.28, 255.34, 254.77, 253.46, 245.28, 248.67, 248.93, 246.46, 245.92, 244.06, 228.99, 230.52, 224.85, 229.12, 224.3, 223.34]","LINESTRING (1730788.209 5908674.963, 1730452.832 5908892.959)"
8,100632739711,Au

,Region,AOI,Date,Distance,Total_UNCY,SourceFile,geometry,Unique_ID,YearsSinceBase,NSM,EPR,LRR,WLR
0,Auckland,KarekareBethells,1960-08-19,251.699492,3.763075,<NA>,POINT Z (1730600.339 5908699.234 0),100632729406,0.000000,-20.13,-0.33,-0.30,-0.30
1,Auckland,KarekareBethells,1988-03-20,240.072177,4.731338,<NA>,POINT Z (1730592.855 5908708.133 0),100632729406,27.583847,-20.13,-0.33,-0.30,-0.30
2,Auckland,KarekareBethells,2004-01-02,248.271204,10.054614,<NA>,POINT Z (1730598.132 5908701.858 0),100632729406,43.370294,-20.13,-0.33,-0.30,-0.30
3,Auckland,KarekareBethells,2011-01-03,237.160591,8.590327,<NA>,POINT Z (1730590.981 5908710.361 0),100632729406,50.373717,-20.13,-0.33,-0.30,-0.30
4,Auckland,KarekareBethells,2011-09-23,241.810177,2.276247,<NA>,POINT Z (1730593.974 5908706.803 0),100632729406,51.093771,-20.13,-0.33,-0.30,-0.30
5,Auckland,KarekareBethells,2015-03-01,237.357058,2.935183,<NA>,POINT Z (1730591.108 5908710.211 0),100632729406,54.529774,-20.13,-0.33,-0.30,-0.30
6,Auckland,KarekareBethells,2016-04-05,231.394650,2.295931,<NA>,POINT Z (1730587.271 5908714.774 0),100632729406,55.627652,-20.13,-0.33,-0.30,-0.30
7,Auckland,KarekareBethells,2017-01-04,232.092825,2.935183,<NA>,POINT Z (1730587.72 5908714.24 0),100632729406,56.377823,-20.13,-0.33,-0.30,-0.30
8,Auckland,KarekareBethells,2021-03-21,231.570784,8.837170,<NA>,POINT Z (1730587.384 5908714.64 0),100632729406,60.585900,-20.13,-0.33,-0.30,-0.30
9,Auckland,KarekareBethells,1940-04-14,263.072720,4.191919,<NA>,POINT Z (1730615.312 5908696.965 0),100632730071,0.000000,-38.94,-0.48,-0.46,-0.51


Rates rows: 25,808
Point rows: 297,097
DSAS exclusions rows: 599
time: 55min 13s (started: 2026-08-11 15:42:25 +12:00)


In [11]:
# 5) Save DSAS outputs
rates.to_file(RATES_OUT)
points.to_file(POINTS_OUT)

# Save exclusions report (shorelines not included in DSAS and why).
if 'dsas_exclusions' in globals():
    dsas_exclusions.to_csv(EXCLUSIONS_OUT, index=False)
else:
    pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason']).to_csv(EXCLUSIONS_OUT, index=False)

# Optional tabular exports for easy QA
rates_csv = RATES_OUT.with_suffix('.csv')
points_csv = POINTS_OUT.with_suffix('.csv')
rates.drop(columns='geometry').to_csv(rates_csv, index=False)
points.drop(columns='geometry').to_csv(points_csv, index=False)

print(f"Saved shorelines shapefile: {V2_PATH}")
print(f"Saved rates shapefile: {RATES_OUT}")
print(f"Saved points shapefile: {POINTS_OUT}")
print(f"Saved exclusions report: {EXCLUSIONS_OUT}")
print(f"Saved CSV companions: {rates_csv.name}, {points_csv.name}")


c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value '['1955-08-09', '1980-10-24', '1988-03-20', '2006-08-23', '2006-08-23', '2006-08-23', '2008-09-20', '2008-09-20', '2010-04-07', '2010-04-07', '2011-01-03', '2011-01-03', '2013-03-25', '2013-03-25', '2015-03-01', '2015-03-01', '2017-01-04', '2017-01-04', '2021-03-21', '2021-03-21']' of field Dates has been truncated to 254 characters.  This warning will not be emitted any more for that layer.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'YearsSinceBase' to 'YearsSince'
  ogr_write(


Saved shorelines shapefile: DataUpdatev2\catriona\NZCCDv2_since20240718.shp
Saved rates shapefile: DataUpdatev2\catriona\ratesv2_since20240718.shp
Saved points shapefile: DataUpdatev2\catriona\intersectsv2_since20240718.shp
Saved exclusions report: DataUpdatev2\catriona\new_dsas_exclusions_since20240718.csv
Saved CSV companions: ratesv2_since20240718.csv, intersectsv2_since20240718.csv
time: 9.06 s (started: 2026-08-11 16:37:39 +12:00)
